# DeepTrack2 dogfooding: does QAChat answer questions about a third-party codebase correctly?

Same harness as `80_qachat_dogfood_code.ipynb`, pointed at a different corpus. That notebook asked
whether QAChat's real end-to-end chat pipeline -- retrieval, prompt construction, generation, citation
-- honors its own system prompt's contract when asked about QAChat's own source. This one asks the same
question about a codebase QAChat had no part in writing: [DeepTrack2](https://github.com/DeepTrackAI/DeepTrack2),
a modular Python library for image data pipelines used in microscopy and ML.

**Why this corpus.** DeepTrack2 is pure Python and close in scale to QAChat's own package -- the right
size for a hand-authored, human-verified gold set rather than something needing dozens of questions to
say anything meaningful. It's also a real codebase with a real domain expert available to write ground
truth: the gold questions and fact checklists below need someone who actually knows this code, the same
way notebook 80's did.

**The corpus covers the `deeptrack/` package *and* the `tutorials/` notebooks, not the package alone.**
DeepTrack2 ships an extensive `tutorials/` tree (getting-started, real-world examples, advanced topics,
developer guides -- all `.ipynb`) alongside the `deeptrack/` package itself, and a lot of the real,
intended-usage knowledge (how a module is actually meant to be used, not just its docstring) lives
there rather than in the package's own source. QAChat already parses `.ipynb` natively (code and
Markdown cells, outputs stripped) and includes it by default for a "code" kind project -- Section 4
below points the indexed source at the whole repository root rather than just `deeptrack/`, so the
tutorials are actually included, not just present on disk.

**The gold set is currently empty.** `benchmarks/gold/dogfood_deeptrack_code.json` has the same schema
as notebook 80's (`task_id`, `question`, `gold_citations`, `fact_checklist`, `requires_abstention`,
`reference_answer`) with zero tasks in it -- this notebook runs end-to-end against an empty question
set until tasks are added. See notebook 80's gold file for the format to follow, and its intro cell for
what Tier 1 (mechanical) vs. Tier 2 (hand-graded) actually check. Worth writing some gold questions that
specifically target `tutorials/` content (documented usage, worked parameter choices) rather than only
questions answerable from `deeptrack/` source alone -- that's the main thing this corpus can test that
notebook 80's couldn't.

**What's different from notebook 80.** QAChat's own dogfood notebook opens `qachat/` from inside this
same repo. DeepTrack2 lives in its own separate checkout, so Section 2 below needs a real local path to
one -- there's no dataset to download and no default that works for everyone. Section 6's
stronger-system-prompt override is carried over as a working default from notebook 80's own tuning
(see that notebook's Section 5 for the history), not re-derived here; if DeepTrack2's results suggest
it needs different tuning, that's worth revisiting once real gold questions exist to test against.

## 1. Prepare QAChat

Locate the QAChat repo root, put it on the import path, and run QAChat's notebook setup. This notebook
still runs QAChat's own benchmarking harness (`benchmarks/common.py`) from inside this repo -- only the
corpus it points QAChat *at* is external.

In [ ]:
from pathlib import Path

root = Path.cwd()
while not (root / "qachat").is_dir() and root != root.parent:
    root = root.parent

print(f"QAChat root: {root}")

In [ ]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [ ]:
from setup.notebook import prepare_notebook

root = prepare_notebook(root)

## 2. Point at a local DeepTrack2 checkout

Unlike `qachat/`, DeepTrack2 isn't part of this repo -- set `DEEPTRACK_SOURCE` below to wherever your
own checkout lives. The default guess assumes it sits beside this repo (`../DeepTrack2`, the common
layout when both are cloned under the same GitHub folder); edit it if yours is somewhere else. This
should be the repo root (containing both `deeptrack/` and `tutorials/`), not the `deeptrack/` package
subfolder -- Section 4 indexes the whole thing.

In [ ]:
DEEPTRACK_SOURCE = root.parent / "DeepTrack2"  # <-- edit if your checkout lives elsewhere

if not DEEPTRACK_SOURCE.is_dir():
    raise FileNotFoundError(
        f"No DeepTrack2 checkout found at {DEEPTRACK_SOURCE}. "
        "Set DEEPTRACK_SOURCE above to your local clone of DeepTrackAI/DeepTrack2."
    )

deeptrack_package = DEEPTRACK_SOURCE / "deeptrack"
if not deeptrack_package.is_dir():
    raise FileNotFoundError(
        f"{DEEPTRACK_SOURCE} doesn't contain a deeptrack/ package -- "
        "check DEEPTRACK_SOURCE points at the repo root, not a subfolder."
    )

tutorials_dir = DEEPTRACK_SOURCE / "tutorials"
if not tutorials_dir.is_dir():
    print(f"NOTE: no tutorials/ folder found at {tutorials_dir} -- indexing deeptrack/ only.")

print(f"DeepTrack2 checkout: {DEEPTRACK_SOURCE}")
print(f"Package: {deeptrack_package}  (exists)")
print(f"Tutorials: {tutorials_dir}  ({'exists' if tutorials_dir.is_dir() else 'missing'})")

## 3. Load the gold question set

Same loader as notebook 80 (`load_generation_tasks`, `benchmarks/common.py`) -- it's corpus-agnostic,
only the schema matters. Starts empty; add tasks to `benchmarks/gold/dogfood_deeptrack_code.json` in
notebook 80's gold file's format.

In [ ]:
from benchmarks.common import load_generation_tasks

gold_path = root / "benchmarks" / "gold" / "dogfood_deeptrack_code.json"
tasks = load_generation_tasks(gold_path)

abstention_count = sum(1 for task in tasks if task.requires_abstention)
print(f"Loaded {len(tasks)} questions ({abstention_count} deliberate abstention questions).")
for task in tasks:
    marker = " [abstention]" if task.requires_abstention else ""
    print(f"  {task.task_id}{marker}: {task.question}")

if not tasks:
    print("\nGold set is empty -- add tasks to", gold_path.relative_to(root), "before this is a real benchmark.")

## 4. Open DeepTrack2 as a real QAChat project

Same real function notebook 80 uses (`prepare_project_workspace`), pointed at the whole `DEEPTRACK_SOURCE`
repo root so both `deeptrack/` and `tutorials/` get mirrored -- not just the package. Mirrors it into a
managed workspace under `benchmarks/sandbox/`, tracked with a SHA-256 manifest -- reopening after a
local DeepTrack2 change picks up only what actually changed.

`.ipynb` is on by default for a "code" kind project (QAChat parses notebooks natively -- code and
Markdown cells, outputs stripped -- and a `tutorials/`-heavy repo like this one is exactly why that's
the default, not an opt-in). Two settings still need adjusting from the default, both applied via
QAChat's own `configure_project` (not a raw file edit):

- The default 1 MB per-file limit is sized for source files, not notebooks with embedded plot
  outputs -- several `tutorials/2-examples` and `3-advanced-topics` notebooks are likely to exceed it
  and get silently skipped even though their actual code+Markdown content is small.
- A brand-new workspace has no prior `profile.ini`, so it falls back to `project.py`'s hardcoded
  default chat model -- the small, fast 0.6B one. Fine as a general-purpose default, but too weak for
  this benchmark's answers to be representative. Enforced unconditionally here, the same way
  notebook 80 does for QAChat itself, so a fresh workspace and a reused one both end up on the same
  known-good model rather than silently diverging.

In [ ]:
from qachat.project import configure_project, prepare_project_workspace

workspace_root = root / "benchmarks" / "sandbox"
project = prepare_project_workspace(
    source_directory=DEEPTRACK_SOURCE,
    kind="code",
    workspace_root=workspace_root,
)

DESIRED_MAX_FILE_BYTES = 5 * 1024 * 1024  # 5 MB -- generous for a notebook with a few plots

if project.profile.max_file_bytes != DESIRED_MAX_FILE_BYTES:
    project = configure_project(
        project.root,
        {"max_file_bytes": DESIRED_MAX_FILE_BYTES},
    )
    # Setting changed after the mirror above was built under the old limit --
    # resync now so previously-skipped-for-size files actually get copied in.
    project = prepare_project_workspace(
        source_directory=DEEPTRACK_SOURCE,
        kind="code",
        workspace_root=workspace_root,
    )

# A brand-new workspace (no prior profile.ini) falls back to project.py's
# hardcoded defaults, which pick the small 0.6B chat model -- fine as a safe
# default generally, but much weaker than what this benchmark expects.
# Enforced unconditionally, the same way notebook 80 does for QAChat itself,
# so a fresh workspace and a reused one both end up on the same model.
DESIRED_MODEL_SETTINGS = {
    "chat_mlx": "Qwen/Qwen3-8B-MLX-4bit",
    "chat_torch": "Qwen/Qwen3-8B",
    "embedding_mlx": "mlx-community/Qwen3-Embedding-0.6B-4bit-DWQ",
    "embedding_torch": "Qwen/Qwen3-Embedding-0.6B",
}
if any(
    getattr(project.profile, key) != value
    for key, value in DESIRED_MODEL_SETTINGS.items()
):
    project = configure_project(project.root, DESIRED_MODEL_SETTINGS)
    print("Model settings corrected to this benchmark's known-good configuration.")

print(f"Project root: {project.root.relative_to(root)}")
print(f"Project source: {project.source}")
print(f"Extensions indexed: {project.profile.extensions}")
print(f"Max file size: {project.profile.max_file_bytes:,} bytes")
print(f"Chat model (mlx): {project.profile.chat_mlx}")
print(f"Embedding model (mlx): {project.profile.embedding_mlx}")
print(f"Chunking: target={project.profile.target_tokens} max={project.profile.max_tokens} overlap={project.profile.overlap_tokens}")


## 5. Open a real RAGSession

Loads both models and builds (or reuses) the semantic, lexical, and code-graph indexes over DeepTrack2.
`progress_callback` prints QAChat's own structured progress events so a first run (no cached index yet)
doesn't sit silently -- likely slower than notebook 80's, since this is indexing a codebase QAChat has
never seen before.

In [ ]:
from qachat.rag import RAGSession

def _print_progress(event):
    if event.stage in {"ready"} or event.current is None:
        print(f"[{event.percent:5.1f}%] {event.message}")

rag_session = RAGSession(project.root, progress_callback=_print_progress)

print(f"Backend: {rag_session.backend}")
print(f"Index: {rag_session.index_status}")
print(f"Chunks: {len(rag_session.chunks)}")

## 6. Use the stronger abstention instruction (carried over default)

Notebook 80's Section 5 found QAChat's default system prompt failed every abstention question until a
more explicit, fixed-token instruction was substituted in -- a prompt-following gap, not a raw
capability gap. That fix is carried over here as the current best-known default rather than
re-discovered from scratch; toggle `USE_STRONGER_PROMPT` off to test DeepTrack2 against QAChat's
original shipped prompt instead once there are enough gold questions to tell the difference.

In [ ]:
USE_STRONGER_PROMPT = True  # toggle: True = override with the instruction below, False = QAChat's original _SYSTEM_PROMPT

_STRONGER_SYSTEM_PROMPT = (
    "Answer only from the supplied project evidence. Put source numbers such "
    "as [1] and [2] directly after supported claims. Do not write a separate "
    "reference list or copy bibliographic text. If the evidence clearly does not "
    "address the question, respond with exactly: Insufficient evidence. "
    "Do not guess or answer from general knowledge."
)

if USE_STRONGER_PROMPT:
    rag_session.chat_session._system_prompt = _STRONGER_SYSTEM_PROMPT
    print("Chat session system prompt overridden for this run:")
    print(_STRONGER_SYSTEM_PROMPT)
else:
    print("USE_STRONGER_PROMPT is False -- using QAChat's original _SYSTEM_PROMPT for this run:")
    print(rag_session.chat_session._system_prompt)

## 7. Check gold citations against the real, chunked corpus

Confirms each question's gold citation(s) actually land inside a real chunk before asking anything --
catches a stale line number (e.g. after a DeepTrack2 update) before every question has already run.

In [ ]:
from benchmarks.common import gold_evidence_indices

class _ChunkAsEvidence:
    """Adapt a real Chunk to the same (path, start_line, end_line, cell,
    citation) shape gold_evidence_indices expects from an Evidence --
    citation is unused for this check, only path/cell/line overlap matters.
    cell matters for a notebook-derived chunk (tutorials/*.ipynb): every
    cell's line numbers start at 1, so a gold citation naming a specific
    cell ("path#cell:1-8") needs the real cell number to disambiguate, not
    just the line range.
    """

    __slots__ = ("citation", "path", "start_line", "end_line", "cell")

    def __init__(self, chunk):
        self.citation = 0
        self.path = chunk.path
        self.start_line = chunk.start_line
        self.end_line = chunk.end_line
        self.cell = chunk.cell


chunk_evidence = [_ChunkAsEvidence(chunk) for chunk in rag_session.chunks]

unmatched = []
for task in tasks:
    if task.requires_abstention:
        continue
    if not gold_evidence_indices(chunk_evidence, task.gold_citations):
        unmatched.append(task.task_id)

if unmatched:
    print(f"WARNING: {len(unmatched)} question(s) have a gold citation that matches no real chunk: {unmatched}")
    print("Check these against the current deeptrack/ or tutorials/ source before trusting their scores below.")
else:
    answerable = len(tasks) - sum(1 for t in tasks if t.requires_abstention)
    print(f"All {answerable} answerable questions' gold citations match a real chunk.")

## 8. Ask every question for real

`rag_session.reset()` before each question keeps every turn an independent, single-turn conversation.
`rag_session.turns[-1].evidence` after `ask()` is the exact, real `Evidence` QAChat retrieved and
supplied for that answer.

In [ ]:
from tqdm.auto import tqdm

run_results: list[dict[str, object]] = []

for task in tqdm(tasks, desc="Asking", unit="question"):
    rag_session.reset()
    answer = rag_session.ask(task.question)
    turn = rag_session.turns[-1]
    run_results.append(
        {
            "task_id": task.task_id,
            "question": task.question,
            "answer": answer,
            "evidence": [
                {
                    "citation": item.citation,
                    "path": str(item.path),
                    "start_line": item.start_line,
                    "end_line": item.end_line,
                    "symbol": item.symbol,
                    "text": item.text,
                    "cell": item.cell,
                }
                for item in turn.evidence
            ],
        }
    )

print(f"Asked {len(run_results)} questions.")

## 9. Score Tier 1 (mechanical)

Same mechanics as notebook 80: citation validity, citation coverage, abstention correctness, and
verbatim-copy detection. `QUOTE_EXEMPT_TASK_IDS` generalizes notebook 80's A3 special case (the one
question there where quoting the source verbatim was the correct answer, not a violation) -- empty
until a DeepTrack2 question needs the same exemption.

In [ ]:
from dataclasses import dataclass as _dataclass
from pathlib import Path as _Path

from benchmarks.common import (
    abstains,
    citation_coverage,
    citation_validity,
    verbatim_overlap_chars,
)

VERBATIM_CHAR_THRESHOLD = 60  # a shorter shared run is plausibly just a common phrase/identifier
QUOTE_EXEMPT_TASK_IDS: frozenset[str] = frozenset()  # task_ids where verbatim quoting is the correct answer


@_dataclass
class _EvidenceView:
    citation: int
    path: _Path
    start_line: int
    end_line: int
    symbol: object
    text: str
    cell: int | None = None


tasks_by_id = {task.task_id: task for task in tasks}
tier1_rows: list[dict[str, object]] = []

for row in run_results:
    task = tasks_by_id[row["task_id"]]
    evidence = [_EvidenceView(**item) for item in row["evidence"]]

    valid, invalid_indices = citation_validity(row["answer"], evidence_count=len(evidence))
    coverage = (
        None
        if task.requires_abstention
        else citation_coverage(row["answer"], evidence, task.gold_citations)
    )
    abstained = abstains(row["answer"])
    verbatim_chars = (
        None
        if task.task_id in QUOTE_EXEMPT_TASK_IDS
        else verbatim_overlap_chars(row["answer"], evidence)
    )

    tier1_rows.append(
        {
            "task_id": task.task_id,
            "requires_abstention": task.requires_abstention,
            "citation_valid": valid,
            "invalid_citation_indices": invalid_indices,
            "citation_coverage": coverage,
            "abstained": abstained,
            "abstention_correct": (
                (abstained if task.requires_abstention else not abstained)
            ),
            "verbatim_overlap_chars": verbatim_chars,
            "verbatim_flag": (
                verbatim_chars is not None and verbatim_chars >= VERBATIM_CHAR_THRESHOLD
            ),
        }
    )

valid_count = sum(1 for r in tier1_rows if r["citation_valid"])
abstention_correct_count = sum(1 for r in tier1_rows if r["abstention_correct"])
coverage_values = [r["citation_coverage"] for r in tier1_rows if r["citation_coverage"] is not None]
verbatim_flags = [r["task_id"] for r in tier1_rows if r["verbatim_flag"]]

print(f"Citation validity: {valid_count}/{len(tier1_rows)} answers cite only real evidence indices.")
print(f"Abstention correctness: {abstention_correct_count}/{len(tier1_rows)} ({abstention_count} of these are the deliberate negatives).")
if coverage_values:
    print(f"Mean citation coverage (answerable questions with retrieved gold evidence, n={len(coverage_values)}): {sum(coverage_values) / len(coverage_values):.2f}")
else:
    print("No answerable question had its gold evidence retrieved -- coverage undefined.")
print(f"Verbatim-copy flags (>= {VERBATIM_CHAR_THRESHOLD} chars): {verbatim_flags or 'none'}")

## 10. Export the Tier 2 grading worksheet

Same as notebook 80: faithfulness and fact-checklist satisfaction need a human reading the answer, not
a regex -- no LLM-judge API call wired in here, by design.

In [ ]:
worksheet_lines = []
for row, tier1 in zip(run_results, tier1_rows):
    task = tasks_by_id[row["task_id"]]
    worksheet_lines.append(f"## {task.task_id}{' [abstention]' if task.requires_abstention else ''}")
    worksheet_lines.append(f"**Question:** {task.question}")
    worksheet_lines.append("")
    worksheet_lines.append(f"**Answer:**\n{row['answer']}")
    worksheet_lines.append("")
    worksheet_lines.append("**Evidence supplied:**")
    for item in row["evidence"]:
        symbol = f" ({item['symbol']})" if item["symbol"] else ""
        worksheet_lines.append(f"- [{item['citation']}] {item['path']}:{item['start_line']}-{item['end_line']}{symbol}")
    worksheet_lines.append("")
    worksheet_lines.append("**Fact checklist:**")
    for fact in task.fact_checklist:
        worksheet_lines.append(f"- [ ] {fact}")
    worksheet_lines.append("")
    if task.reference_answer:
        worksheet_lines.append(
            "**Reference reply (from the fact checklist -- a comparison "
            "aid, not ground truth on its own):**"
        )
        worksheet_lines.append(task.reference_answer)
        worksheet_lines.append("")
    worksheet_lines.append(
        f"**Tier 1 (automatic):** citation_valid={tier1['citation_valid']}, "
        f"citation_coverage={tier1['citation_coverage']}, abstained={tier1['abstained']}, "
        f"abstention_correct={tier1['abstention_correct']}, "
        f"verbatim_overlap_chars={tier1['verbatim_overlap_chars']}"
    )
    worksheet_lines.append("")
    worksheet_lines.append("**Tier 2 (your read):** faithful? [ ]  checklist satisfied? [ ]  notes: ")
    worksheet_lines.append("\n---\n")

worksheet_text = "\n".join(worksheet_lines)
worksheet_path = root / "benchmarks" / "results"
worksheet_path.mkdir(parents=True, exist_ok=True)

## 11. Save results

Same pattern as notebook 80, distinguished by filename (`dogfood_deeptrack_code` instead of
`dogfood_code`) so results from the two corpora never collide in `benchmarks/results/`.

In [ ]:
import json as _json
from benchmarks.common import local_run_timestamp

results_dir = root / "benchmarks" / "results"
results_dir.mkdir(parents=True, exist_ok=True)

timestamp, stamp = local_run_timestamp()

run_record = {
    "benchmark": "QAChat dogfooding -- DeepTrack2, hand-authored gold questions (Area A: code)",
    "timestamp": timestamp,
    "corpus_source": str(DEEPTRACK_SOURCE),
    "chat_model": project.profile.chat_model(rag_session.backend),
    "embedding_model": project.profile.embedding_model(rag_session.backend),
    "backend": rag_session.backend,
    "index_status": rag_session.index_status,
    "chunk_count": len(rag_session.chunks),
    "question_count": len(tasks),
    "abstention_question_count": sum(1 for t in tasks if t.requires_abstention),
    "gold_citation_unmatched": unmatched,
    "system_prompt": rag_session.chat_session._system_prompt,
    "tier1_summary": {
        "citation_valid_count": valid_count,
        "abstention_correct_count": abstention_correct_count,
        "mean_citation_coverage": (
            sum(coverage_values) / len(coverage_values) if coverage_values else None
        ),
        "verbatim_flagged_task_ids": verbatim_flags,
    },
    "per_question": [
        {**row, "tier1": tier1}
        for row, tier1 in zip(run_results, tier1_rows)
    ],
}

output_path = results_dir / f"{stamp}_dogfood_deeptrack_code.json"
output_path.write_text(_json.dumps(run_record, indent=2))

worksheet_output_path = results_dir / f"{stamp}_dogfood_deeptrack_code_tier2_worksheet.md"
worksheet_output_path.write_text(worksheet_text)

print(f"Saved: {output_path.relative_to(root)}")
print(f"Saved: {worksheet_output_path.relative_to(root)}")

## 12. Release the models

In [ ]:
rag_session.close()

print("Chat and embedding models released.")